# Example Usage - Differential Evolution (DE)

Auto Reload Submodules

In [ ]:
%load_ext autoreload
%autoreload 2

Import path

In [ ]:
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

Set up logger

In [ ]:
import logging

logging.basicConfig(
    encoding="utf-8",
    filemode="a",
    format="[{asctime}] [{levelname}] {message}",
    style="{",
    datefmt="%Y-%m-%d, %H:%M",
    level=logging.INFO,
)

logger = logging.getLogger()
logger.info("Hello logging!")

Lets First define a problem:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

x = np.linspace(0, 10, 500)
y = np.cos(x) + rng.normal(0, 0.2, 500)

In [ ]:
def rmse(y, y_pred):
    return np.sqrt(sum((y - y_pred)**2) / len(y))

Next, lets make a population which we will evolve:

In [ ]:
from pyeas._population import Genes, Population

member = [
    Genes(bounds=(-5,5), number=6),
]

pop = Population(
    size=40,
    member=member,
    seed=42,
)

pop

In [ ]:
pop.population[0][0]

In [ ]:
pop._normalised_population[0][0]

In [ ]:
pop.denormalise(pop._normalised_population)

### The First Generation 

Now, lets initialize our solver, and iterate though the fist generation of population members:

In [ ]:
from pyeas._de import DE 

optimizer = DE(
    population=pop,
    mut=0.6,
    crossp=0.6,
    mut_scheme = 'ttb1',  # 'ttb1', rand1
    seed=1,
)

In [ ]:
trial_pop = optimizer.ask(loop=0)
print(np.shape(trial_pop))

trial_pop[:3]

In [ ]:
from examples.funcs import polynomial_order_5

solutions = []
for t, trial in enumerate(trial_pop):
    
    pred = polynomial_order_5(x, trial)
    value = rmse(y, pred)
    # print(t, value)

    solutions.append((value))

solutions[:5]

In [ ]:
optimizer.tell(solutions, trial_pop)

In [ ]:
optimizer.population.population[optimizer._best_idx]

### Generational Optimization

Starting from scratch:

In [ ]:
from pyeas._de import DE 

optimizer = DE(
    population=pop,
    mut=0.4,
    crossp=0.4,
    mut_scheme = 'best2',  # 'ttb1', rand1
    seed=1,
)

In [ ]:
from tqdm import tqdm
from examples.funcs import polynomial_order_5

num_gens = 2000

pbar = tqdm(range(num_gens), unit=' generations')
for generation in pbar:

    # Ask a parameter
    trial_pop = optimizer.ask(loop=generation)

    solutions = []
    for trial in trial_pop:
        
        pred = polynomial_order_5(x, trial)

        value = rmse(y, pred)

        # value = q3(trial[0], trial[1], trial[2])
        solutions.append((value))

    # Tell evaluation values.
    optimizer.tell(solutions, trial_pop)
    
    pbar.set_description_str(f"Best Member: {optimizer.best_member[1]}, loss: {optimizer.best_member[0]:.4} ")


In [ ]:
import matplotlib.pyplot as plt

# # Plot convergence
fig, ax = plt.subplots()
ax.plot(optimizer.history['best_fits'])
plt.yscale("log")

print(optimizer.history['best_solutions'][-1])

In [ ]:
# # Plot final best solution
fig, ax = plt.subplots()
ax.scatter(x, y, marker=".", color='r', alpha=0.7, label='Target data')
plt.plot(x, np.cos(x), '--', label='ideal cos(x)', color='k', alpha=0.5)
data = polynomial_order_5(x, optimizer.history['best_solutions'][-1])
ax.plot(x, data, label='DE Solution')
ax.legend()

In [ ]:
import matplotlib.animation as animation
# %matplotlib notebook
plt.rcParams["animation.html"] = "jshtml"

# # Plot Ani
fig_ani, (ax, ax2) = plt.subplots(ncols=2, figsize=(9,4))

fig_ani.suptitle('DE fitting a 5th order polynomial to noisy cos() data')

ax.set_ylim([-10, 10])
ax.scatter(x, y, marker=".", color='r')
ax.set_xlabel("x")
ax.set_ylabel("y")

ax2.set_yscale('log')
ax2.plot(optimizer.history['best_fits'])
ax2.set_xlabel("Generation")
ax2.set_ylabel("rmse")
it_line, = ax2.plot([0, 0],  [np.min(optimizer.history['best_fits']), np.max(optimizer.history['best_fits'])], markersize=5, color='k', alpha=0.5) #
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

lines = []


def ani(i):
    # ax.clear()

    lim = 10 - (i/num_gens)*(10-4)

    ax.set_ylim([-lim, lim])

    data = polynomial_order_5(x, optimizer.history['best_solutions'][i])
    line, = ax.plot(x, data, alpha=0.4)

    lines.append(line)
    if len(lines) > 10:
        lines[0].remove()
        lines.pop(0)

    # ax.set_title("Generation %d" % (i))

    it_line.set_xdata([i, i])

length = 20 # seconds
FPS = num_gens/length  # 20
the_animation = animation.FuncAnimation(fig_ani, ani, frames=np.arange(num_gens), interval=20)

# fig_path = "examples/DE.gif"
# the_animation.save(fig_path, writer='pillow', fps=FPS, dpi=50)


In [ ]:
the_animation